# M7 checker baselines on Colab T4: zero-shot DeBERTa and Qwen3-8B judge (via the project gateway)

**What this runs:** the merged baseline runner (`python -m app.evaluation checker-baselines`) against the **real project gateway** on this T4, over **loopback only** (no tunnel), as in the M3 and M5 acceptance runs:
- `--scorer deberta`: the pinned `MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli@6f5cf0a2` via `/check`, at the configured `CHECKER_ENTAILMENT_THRESHOLD` (0.5, uncalibrated, not selected).
- `--scorer qwen-judge`: `Qwen/Qwen3-8B-AWQ@4da05a8e` via the chat route, prompt `m7-judge-1`, one-word answer.

Before any pair is sent, the runner refuses fake inference and verifies that the gateway serves the pinned model and revision.

**Inputs:** the 37 human-agreed dev pairs. You upload the two imported label files (`gold-ann-a.csv`, `gold-ann-b.csv`) from your laptop, so the labels reach only this private runtime. **Never upload** the reviewers' filled sheets or the coordinator `.key.json` files; they are not needed. The notebook checks that the dataset revision is `cc05512a3fc4…` and that `inputs_sha256` is `7ad97fc97b42…`, the values verified on the laptop.

**Setup:** *Runtime → Change runtime type → T4 GPU*, on a fresh runtime. Run the cells in order; it takes about 25–35 minutes, mostly installing the pinned vLLM environment and loading the models.

**Return this file:** `m7-baselines-gateway-<timestamp>.zip`, downloaded by the last cell. It holds the two runner reports, the gateway `/health`, the export and the runtime versions. It contains no label CSVs.

In [ ]:
# @title 1 · Clone the repository and check the GPU (must be a Tesla T4)
import os, subprocess
REPO = "/content/Enterprise-AI-OS"
REF = "main"  # or the exact commit from the PR
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "https://github.com/youssefhatembahig6-alt/Enterprise-AI-OS", REPO], check=True)
subprocess.run(["git", "-C", REPO, "checkout", "-q", REF], check=True)
subprocess.run(["git", "-C", REPO, "log", "-1", "--format=%H %s"], check=True)
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True, check=True).stdout.strip()
print(gpu)
assert gpu.startswith("Tesla T4"), "select a T4 GPU runtime"

In [ ]:
# @title 2 · Install the hash-locked backend environment (Python 3.13, as in M3 and M5)
subprocess.run("pip -q install uv==0.9.5", shell=True, check=True)
if not os.path.exists("/content/venv-backend/bin/python"):
    subprocess.run("uv venv -q --python 3.13 /content/venv-backend", shell=True, check=True)
subprocess.run(f"uv pip install -q --python /content/venv-backend/bin/python --require-hashes -r {REPO}/backend/requirements.lock",
               shell=True, check=True)
print("backend venv ready")

In [ ]:
# @title 3 · Upload the two imported label files (kept outside the repository), verify the export
import hashlib, json
from google.colab import files
LABEL_DIR = "/content/m7-labels"  # private to this runtime; passed to the runner with --label-dir
os.makedirs(LABEL_DIR, exist_ok=True)
uploaded = files.upload()  # choose gold-ann-a.csv and gold-ann-b.csv (the imported label files)
assert sorted(uploaded) == ["gold-ann-a.csv", "gold-ann-b.csv"], sorted(uploaded)
for name, data in uploaded.items():
    open(f"{LABEL_DIR}/{name}", "wb").write(data)
os.makedirs("/content/m7-baselines", exist_ok=True)
PY = "/content/venv-backend/bin/python"
EXPORT = "/content/m7-baselines/m7-checker-inputs.jsonl"
out = subprocess.run([PY, "-m", "app.evaluation", "checker-baselines", "--label-dir", LABEL_DIR,
                      "--export-inputs", EXPORT], cwd=f"{REPO}/backend", capture_output=True, text=True)
print(out.stdout, out.stderr)
head = json.loads(open(EXPORT).readline())
assert head["pairs"] == 37, head
assert head["inputs_sha256"] == "7ad97fc97b42ecc213315d452964dcf2bff00d12a5c46cfa3a22b8adcbf48f9b", head
assert hashlib.sha256(open(EXPORT, "rb").read()).hexdigest() == "97bd0387058b0d2aab006bee84b11a43fe4d38e20e85c5a32f2f9b205c59c1ec"
print("verified: 37 agreed dev pairs, inputs", head["inputs_sha256"][:12])

In [ ]:
# @title 4 · Install the pinned vLLM environment (M1 pins: vllm 0.11.0, transformers 4.56.2)
subprocess.run(["python", f"{REPO}/ml/colab-m5/run_m5_code_colab.py", "vllm-env"], check=True)

In [ ]:
# @title 5 · Launch the model gateway on loopback (127.0.0.1:8100; the token never leaves this runtime)
import secrets
os.environ.setdefault("GATEWAY_TOKEN", secrets.token_urlsafe(32))
%run /content/Enterprise-AI-OS/ml/model-gateway/colab_launch.py

In [ ]:
# @title 6 · Score the two gateway baselines with the merged runner
import time
env = dict(os.environ, MODEL_MODE="gateway", LLM_BASE_URL="http://127.0.0.1:8100",
           MODEL_GATEWAY_TOKEN=os.environ["GATEWAY_TOKEN"])
health = subprocess.run(["curl", "-s", "http://127.0.0.1:8100/health"], capture_output=True, text=True, check=True).stdout
open("/content/m7-baselines/gateway-health.json", "w").write(health)
print(health)
codes = {}
for scorer in ("deberta", "qwen-judge"):
    t0 = time.time()
    done = subprocess.run([PY, "-m", "app.evaluation", "checker-baselines", "--scorer", scorer,
                           "--label-dir", LABEL_DIR, "--out", f"/content/m7-baselines/{scorer}.json"],
                          cwd=f"{REPO}/backend", env=env, capture_output=True, text=True)
    codes[scorer] = done.returncode
    print(f"== {scorer}: exit {done.returncode} in {time.time() - t0:.0f} s\n{done.stdout}\n{done.stderr[-2000:]}")
assert codes == {"deberta": 0, "qwen-judge": 0}, codes  # 3 = blocked, 4 = backend rejected
for scorer in ("deberta", "qwen-judge"):
    report = json.load(open(f"/content/m7-baselines/{scorer}.json"))
    assert report["status"] == "scored" and report["dataset"]["human_reviewed_pairs"] == 37
    assert report["dataset"]["dataset_revision"] == "cc05512a3fc4ed6a50823846f7f058bd5386b52875198706bc7345e979db32c6"
    print(scorer, "served:", report["scorer"]["served"]["models"])

In [ ]:
# @title 7 · Bundle the reports (no label CSVs, no token) and download
import zipfile, datetime
versions = {
    "application_commit": subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip(),
    "nvidia_smi": gpu,
    "vllm_env": subprocess.run(["/content/vllm-env/bin/python", "-c", "import vllm, transformers, torch; print(vllm.__version__, transformers.__version__, torch.__version__)"],
                               capture_output=True, text=True).stdout.strip(),
    "backend_python": subprocess.run([PY, "--version"], capture_output=True, text=True).stdout.strip(),
}
json.dump(versions, open("/content/m7-baselines/versions.json", "w"), indent=2)
stamp = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
bundle = f"/content/m7-baselines-gateway-{stamp}.zip"
token = os.environ["GATEWAY_TOKEN"]
with zipfile.ZipFile(bundle, "w") as z:
    for name in ("deberta.json", "qwen-judge.json", "gateway-health.json", "m7-checker-inputs.jsonl", "versions.json"):
        data = open(f"/content/m7-baselines/{name}").read()
        assert token not in data, f"token found in {name}"
        z.writestr(name, data)
files.download(bundle)